In [1]:
import gymnasium as gym
from typing import Optional

We've implemented a gym envorionment for playing blackjack properly... will do models later.
What sort of models are needed to effectively play blackjack given only symbollic knowledge of the game?
We have set up a symbols only environment for the agent to play with, and will see what sort of models we can use to train in this gym.

In [ ]:
class BlackjackTable(gym.Env):
    def __init__(self):
        import numpy as np
        self.observation_space = gym.spaces.Dict(
            {"bet": gym.spaces.Discrete(10**6),
             "bankroll": gym.spaces.Discrete(10**6),
             #i'm actually kinda unsure if we need a bankroll to begin with...
             "playercards": gym.spaces.Sequence(gym.spaces.Discrete(52)),
             "dealercards": gym.spaces.Sequence(gym.spaces.Discrete(52))
            }
        )
        self.action_space = gym.spaces.OneOf((gym.spaces.Box(low = 1, high = 1000, shape = (1,), dtype = np.int64), gym.spaces.Discrete(2)))
        #observe that the action space spits out a tuple. If the first element is 0, it's a bet denoted by element two.
        # if the first is a 1, its a hit/stand, where stand is 0, hit is 1.
        self.deck = np.arange(52) #the deck begins as a sequence of cards
        self.bet = 0 #how much has been bet on the current hand.
        self.bankroll = 0 #this is how much money the player has
        self.position = 0  #this is how far along in the deck the table is. The position is always the next card to be drawn
        self.playercards = () #this is a sequence of cards for the plyayer
        self.dealercards = () #this is a sequnce of cards for the dealer. Ultimately this will only be used to see what the dealer has showing.
        self.threshold = 10 # nmber of cards to shuffle before.
        
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        super().reset(seed=seed)
        self.np_random.shuffle(self.deck)
        self.bankroll = 100
        self.bet = 0 #reset the bet
        self.position = 0  #start at the top
        self.origin = 0 #get ready to move by the top
        #a quick explaination: we'd like to return the cards seen not including the dealt cards...
        
        self.playercards = () #player no cards
        self.dealercards = () #this is a sequnce of cards for the dealer. Ultimately this will only be used to see what the dealer has showing.

    def _get_obs(self):
        return {"bankroll": self.bankroll,
                "bet": self.bet,
                "playercards": self.playercards,
                "dealercards": self.dealercards}

    def _get_info(self):
        return {"pastcards": self.deck[:self.origin]}

    def _reset_board(self, reward): #for resetting the board. gives appropriate reward to bankroll, return for reward
        self.playercards = () 
        self.dealercards = ()
        self.bankroll += reward
        self.bet = 0 #reset bet
        self.origin = self.position
        return reward
        
    def step(self, action):
        import numpy as np

        #we will return these three later.
        terminated = False
        truncated = False
        reward = 0
        
        if ((not self.bet) and (52 - self.position) < self.threshold) or self.bankroll == 0: 
            #end game, since there is no bet and we have limited cards...or no money
            terminated = True
            reward = self.bankroll

        # gave up on this behavior... will keep for posterity
        #'''elif ((not self.bet) and action[0]) or (not action[0] and self.bet) or (not action[0] and (action[1] == 0 or action[1] > self.bankroll)): 
        #    #if the user submits a non-betting action with no cards we do nothing.
        #    #similar if betting action submitted with cards
        #    #and if bet is zero...'''
            
        elif not (self.bet or action[0]): #we've got no cards and they bet properly
            self.playercards = tuple(self.deck[self.position: self.position + 2]) #give the player two cards
            self.dealercards = tuple([self.deck[self.position + 2]]) #give the dealer one card... This is slightly incorrect but whatever.
            self.position += 3
            self.bet = action[1]
            self.bankroll -= self.bet
            reward = -self.bet
            if np.max(BlackjackTable.getValue(self.playercards)) == 21: #if blackjack...
                self.position += 1 #imagine we gave the dealer the last card.
                reward = self._reset_board(np.ceil(self.bet * 2.5))
                
        elif self.bet and action[0]: #we've got a bet and a choice!
            if action[1]: # if hitting...
                self.playercards += (self.deck[self.position],)
                self.position += 1
                reward = 0.5
                if np.min(BlackjackTable.getValue(self.playercards)) > 21: #bust on card
                    reward = self._reset_board(-self.bet)
            else: #if standing
                while np.min(BlackjackTable.getValue(self.dealercards)) < 17: #loop till range is 17
                    self.dealercards += (self.deck[self.position],)
                    self.position += 1
                if np.min(BlackjackTable.getValue(self.dealercards)) > 21: #dealer bust
                    reward = self._reset_board(2 * self.bet)
                else: # we need to compare the values...
                    pv = BlackjackTable.getValue(self.playercards)
                    dv = BlackjackTable.getValue(self.dealercards)
                    pv = pv[-1] if pv[-1] <= 21 else pv[0]
                    dv = dv[-1] if dv[-1] <= 21 else dv[0]
                    if pv == dv:
                        self._reset_board(self.bet) #push
                    elif pv > dv:
                        reward = self._reset_board(2 * self.bet) #player win
                    else:
                        reward = self._reset_board(0)

        return self._get_obs(), reward, terminated, truncated, self._get_info()

    def getSymbolicHand(hand):
        import numpy as np
        handval = np.mod(hand, 13)
        handval[handval == 0] = 10
        handval[handval > 10] = 10
        return handval.astype(np.int64)
        
    def getValue(hand):
        import numpy as np
        handval = BlackjackTable.getSymbolicHand(hand)
        v = np.sum(handval)
        if np.any(handval == 1):
            return [v, v + 10]
            # comments comments comments
        return [v]

In [3]:
def readoutput(output): #reads the output human style...
    import numpy as np
    print("Bankroll is:", output[0]["bankroll"])
    print("Current bet is:", output[0]["bet"])
    print("Player has:", BlackjackTable.getValue(output[0]["playercards"]))
    print("Dealer has:", BlackjackTable.getValue(output[0]["dealercards"]))
    handval = np.mod(output[4]["pastcards"], 13)
    handval[handval == 0] = 13
    print("Previous cards:", handval)
    handval[handval == 1] = 14
    print("Count is", np.sum(handval < 7) - np.sum(handval > 8))
    print("Game Over" if output[2] else "Continue")

In [ ]:
def interprethand(hand): #convert a hand to a vector of instances...will have to flesh it out nicely
    import numpy as np
    v = np.mod(hand, 13).astype(np.int64)
    #return np.unique(v, return_counts = True, sorted = True)
    return np.bincount(v, minlength = 13)